# 01 - Calibrate the replay channel

The paper replays recorded qubit readout traces through a **DAC → coax → ADC loopback** on the RFSoC 4x2, so the
classifier sees them through real converters. The channel changes each trace. To first order the change is affine in the
complex baseband:

$$z_\text{acquired}(t) \approx k\, z_\text{sent}(t) + c,$$

with one complex gain $k$ and one complex offset $c$ per board and bitstream load. Calibration measures $(k, c)$; every
replay afterwards sends the **predistorted** trace $(z - c)/k$, so that what the ADC returns matches the recording.

This notebook runs the two-pass calibration done after every bitstream load (paper Sec. IV):

1. **Pass 1:** send 500 reference envelopes uncorrected and record what comes back.
2. **Fit** $k$ and $c$ by complex least squares between the mean returned trajectory and the mean reference trajectory.
3. **Pass 2:** send the same reference traces predistorted with the fit, and check that they come back on top of the
   reference.

**You need:**
- An RFSoC 4x2 running our firmware and its Pyro server (`host/start_nameserver.ipynb` on the board), with DAC tile 0
  channel 0 cabled to ADC tile 0 channel 0.
- `X_2_padded_11k.npy` (the reference envelopes at the 11,000-sample DAC rate) and `x_50k.npy` (the same traces at the
  770-sample readout rate). Both are listed in `DATA_MANIFEST.csv`.

Nothing is imported from the repository. Results go to `OUT_DIR`.

In [ ]:
from pathlib import Path

BOARD_HOST = "10.1.9.26"   # the board's IP address (our board A: 10.1.9.26, board B: 10.1.4.105)
NS_PORT = 8888              # Pyro name server port used by host/start_nameserver.ipynb
PROXY_NAME = "myqick"       # name the board's QICK object is registered under

DATA_DIR = Path("..")       # where X_2_padded_11k.npy and x_50k.npy are
OUT_DIR = Path("outputs/calibration")
N_SHOTS = 500               # shots per pass (the paper used 500)

OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
import gc
import time

import matplotlib.pyplot as plt
import numpy as np
import Pyro4
from qick import QickConfig
from qick.asm_v2 import AveragerProgramV2
from scipy.interpolate import interp1d

Pyro4.config.SERIALIZER = "pickle"
Pyro4.config.PICKLE_PROTOCOL_VERSION = 4


def connect(host, port=NS_PORT, name=PROXY_NAME, attempts=40, wait_s=3.0):
    # The server may still be registering right after a bitstream load, so retry for a while.
    for i in range(attempts):
        try:
            ns = Pyro4.locateNS(host=host, port=port)
            soc = Pyro4.Proxy(ns.lookup(name))
            return soc, QickConfig(soc.get_cfg())
        except Exception as e:
            if i % 5 == 0:
                print(f"waiting for the board ({type(e).__name__}: {e})")
            time.sleep(wait_s)
    raise RuntimeError("board did not come up")


soc, soccfg = connect(BOARD_HOST)
print(soccfg)

## The loopback program

One shot: play the trace as an arbitrary envelope on generator 0 at a 50 MHz carrier, and record readout 0's decimated
I/Q stream (552.96 MS/s after QICK's FIR and 8x decimation, 1.808 ns per sample).

The trigger also pulses `trig_4`, NN_axi's trigger input (tied off in v1.43 and v1.44, so the pulse has no effect).

In [ ]:
GEN_CH, RO_CH = 0, 0

soccfg["tprocs"][0]["output_pins"].append(("trig", 4, 0, "NN_trigger (trig_4_o)"))
NN_TRIGGER_PIN = len(soccfg["tprocs"][0]["output_pins"]) - 1

BASE_CFG = {
    "gen_ch": GEN_CH, "ro_ch": RO_CH, "nqz": 1, "reps": 1, "final_delay": 100,
    "readout_length": 1.393,   # us of decimated readout to record
    "adc_trig_offset": 0.31,   # us from pulse start to readout trigger
    "freq": 50,                # MHz carrier, mixed down by the readout's DDS
    "mode": "oneshot", "stdysel": "zero", "outsel": "product", "gain": 1, "phase": 0.0,
}


class LoopbackProgram(AveragerProgramV2):
    def __init__(self, soccfg, cfg):
        super().__init__(soccfg, reps=cfg["reps"], final_delay=cfg["final_delay"], cfg=cfg)

    def _initialize(self, cfg):
        self.declare_gen(ch=cfg["gen_ch"], nqz=cfg["nqz"])
        self.declare_readout(ch=cfg["ro_ch"], length=cfg["readout_length"])
        self.add_envelope(ch=cfg["gen_ch"], name="env", idata=cfg["waveform_I"], qdata=cfg["waveform_Q"])
        self.add_pulse(ch=cfg["gen_ch"], name="trace", style="arb", envelope="env", gain=cfg["gain"],
                       freq=cfg["freq"], phase=cfg["phase"], outsel=cfg["outsel"], mode=cfg["mode"],
                       stdysel=cfg["stdysel"], ro_ch=cfg["ro_ch"])
        self.add_readoutconfig(ch=cfg["ro_ch"], name="ro", freq=cfg["freq"], gen_ch=cfg["gen_ch"])

    def _body(self, cfg):
        self.pulse(ch=cfg["gen_ch"], name="trace")
        self.send_readoutconfig(ch=cfg["ro_ch"], name="ro", t=0)
        self.trigger(ros=[cfg["ro_ch"]], pins=[0, NN_TRIGGER_PIN], t=cfg["adc_trig_offset"], width=10)


def pad32(w):
    # Envelopes must be a multiple of 32 values (16 I/Q pairs per fabric word pair).
    w = np.asarray(w, dtype=float).reshape(-1)
    n = int(np.ceil(w.size / 32) * 32)
    return np.pad(w, (0, n - w.size))[:n]


def fire(traces, label):
    # traces: (n, 2*L) interleaved I,Q at the DAC rate. Returns (n, 2*M) interleaved I,Q as recorded.
    rows = []
    for i, tr in enumerate(traces):
        w = pad32(tr)
        cfg = dict(BASE_CFG, waveform_I=w[0::2], waveform_Q=w[1::2])
        iq = LoopbackProgram(soccfg, cfg).acquire_decimated(soc, progress=False)
        rows.append(np.concatenate([np.asarray(a).reshape(-1) for a in iq]) if isinstance(iq, list)
                    else np.asarray(iq).reshape(-1))
        if i % 100 == 0 or i == len(traces) - 1:
            print(f"  {label}: {i + 1}/{len(traces)}")
        if i % 50 == 0:
            gc.collect()
    return np.vstack(rows)


def to_complex(x):
    return x[:, 0::2] + 1j * x[:, 1::2]

## Pass 1: send the reference uncorrected

The reference envelopes are scaled by 8 to fill the DAC range.

In [ ]:
sent_pass1 = np.load(DATA_DIR / "X_2_padded_11k.npy")[:N_SHOTS] * 8
reference = np.load(DATA_DIR / "x_50k.npy")[:N_SHOTS]          # the same traces at the readout rate

t0 = time.time()
acq1 = fire(sent_pass1, "pass 1")
np.save(OUT_DIR / "pass1_acquired.npy", acq1)
print(f"pass 1: {acq1.shape} in {(time.time() - t0) / 60:.1f} min")

## Fit $k$ and $c$

Complex least squares on the **mean** trajectories, sample by sample:
$\bar z_\text{acq}(t) = k\,\bar z_\text{ref}(t) + c$. Averaging over 500 shots removes the noise, so the fit sees only the
channel.

In [ ]:
z_acq, z_ref = to_complex(acq1), to_complex(reference)
m_acq, m_ref = z_acq.mean(axis=0), z_ref.mean(axis=0)
n = min(len(m_acq), len(m_ref))                     # the recording and the reference have the same length (770)
A = np.vstack([m_ref[:n], np.ones(n)]).T
(k, c), *_ = np.linalg.lstsq(A, m_acq[:n], rcond=None)
resid = m_acq[:n] - (k * m_ref[:n] + c)
print(f"k = {k:.5f}   |k| = {abs(k):.5f}   arg(k) = {np.degrees(np.angle(k)):.2f} deg")
print(f"c = {c:.3f}")
# The affine model has no delay or filtering, while the loopback delays the trace by 5-8 samples and low-pass filters it,
# so the residual is not small. k and c only need to get gain, phase and offset right; the plot shows how well they do.
print(f"fit residual rms / signal rms = {np.sqrt(np.mean(abs(resid) ** 2)) / np.sqrt(np.mean(abs(m_acq[:n] - m_acq[:n].mean()) ** 2)):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
t = np.arange(n) * 1.808e-3
for a, part, f in ((ax[0], "I", np.real), (ax[1], "Q", np.imag)):
    a.plot(t, f(m_acq[:n]), label="returned (mean)")
    a.plot(t, f(k * m_ref[:n] + c), "--", label="k·reference + c")
    a.set_xlabel("time (us)"); a.set_title(part)
ax[0].legend(); plt.tight_layout(); plt.show()

## Pass 2: send the reference predistorted

Predistort, $z_\text{sent} = (z_\text{ref} - c)/k$, then interpolate from the 770-sample readout rate up to the 11,000-sample
envelope, and scale to 99% of the DAC's ±32,766 range. A replay (notebook 02) does exactly the same to any trace.

In [ ]:
def predistort(z_rec, k, c, dac_len=11000, fill=0.99, limit=32766):
    # Readout-rate complex traces -> interleaved I,Q envelopes for the DAC. Returns (envelopes, scale factor).
    z = (z_rec - c) / k
    xo, xn = np.linspace(0, 1, z.shape[1]), np.linspace(0, 1, dac_len)
    I = interp1d(xo, z.real, axis=1, kind="linear", fill_value="extrapolate")(xn)
    Q = interp1d(xo, z.imag, axis=1, kind="linear", fill_value="extrapolate")(xn)
    env = np.empty((z.shape[0], 2 * dac_len)); env[:, 0::2] = I; env[:, 1::2] = Q
    scale = limit * fill / np.max(np.abs(env))
    return env * scale, scale


sent_pass2, scale = predistort(z_ref, k, c)
print(f"DAC fill factor {scale:.6f}; peak {np.max(np.abs(sent_pass2)):.1f} of 32766")
acq2 = fire(sent_pass2, "pass 2")
np.save(OUT_DIR / "pass2_acquired.npy", acq2)

## Check

After predistortion the returned mean should sit on the reference, up to the DAC fill factor. That factor scales the
predistorted trace, offset included, so the comparison fits a complex scale and an offset. What remains is the
channel's delay and filtering, the same as in the pass-1 fit.

In [ ]:
m2 = to_complex(acq2).mean(axis=0)[:n]
A2 = np.vstack([m_ref[:n], np.ones(n)]).T
(g, o), *_ = np.linalg.lstsq(A2, m2, rcond=None)          # returned ~ g * reference + o
err = np.sqrt(np.mean(abs(m2 - (g * m_ref[:n] + o)) ** 2)) / np.sqrt(np.mean(abs(m2 - m2.mean()) ** 2))
print(f"pass 2: returned = g * reference + o with g = {abs(g):.4f} at {np.degrees(np.angle(g)):+.2f} deg "
      f"(the phase should be near 0), residual {err:.3f} of the signal")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
for a, part, f in ((ax[0], "I", np.real), (ax[1], "Q", np.imag)):
    a.plot(t, f(m2), label="returned, pass 2 (mean)")
    a.plot(t, f(g * m_ref[:n] + o), "--", label="g·reference + o")
    a.set_xlabel("time (us)"); a.set_title(part)
ax[0].legend(); plt.tight_layout(); plt.show()

In [ ]:
np.save(OUT_DIR / f"k_{BOARD_HOST}.npy", np.complex128(k))
np.save(OUT_DIR / f"c_{BOARD_HOST}.npy", np.complex128(c))
print("saved", OUT_DIR / f"k_{BOARD_HOST}.npy", "and", OUT_DIR / f"c_{BOARD_HOST}.npy")

**Notes**
- $(k, c)$ belongs to one board and one bitstream load. Our two boards differ by about 36° in $\arg k$ with v1.43, and
  replaying with the other board's values costs fidelity. Recalibrate after every load.
- $\arg k$ also moves by about 1° from shot to shot, from the 100 MHz image the FIR passes and a one-sample timing
  shift; averaging over 500 shots is what makes the fit stable (paper Sec. IV).
- The repository's `tools/run_calibration.py` is the script version of this procedure; it stores the pair per board in
  `calib/` for the other scripts.